Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA # Для снижения размерности
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Настройки визуализации
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

Загрузка данных и предобработка

In [ ]:
# Конфигурация путей
BASE_PATH = r'D:/practice/com748_v35_spectrograms/' 
METADATA_PATH = os.path.join(BASE_PATH, 'metadata.csv')
MAX_SAMPLES = 5000 

print(f"Загрузка метаданных из {METADATA_PATH}")
df = pd.read_csv(METADATA_PATH)

# Балансировка классов в выборке (берем поровну fake и real, чтобы модель не училась только на fake)
df_real = df[df['label'] == 0]
df_fake = df[df['label'] == 1]

# Берем половину от MAX_SAMPLES для каждого класса
n_per_class = MAX_SAMPLES // 2
sample_real = df_real.sample(n=n_per_class, random_state=42)
sample_fake = df_fake.sample(n=n_per_class, random_state=42)

df_sample = pd.concat([sample_real, sample_fake]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Сформирована выборка: {len(df_sample)} образцов.")
print(f"Распределение классов: {df_sample['label'].value_counts().to_dict()}")

def load_spectrograms(dataframe, base_path):
    X_data = []
    y_data = []
    
    for idx, row in dataframe.iterrows():
        file_path = os.path.join(base_path, row['filepath'])
        try:
            spectrogram = np.load(file_path)

            flat_vector = spectrogram.flatten()
            
            X_data.append(flat_vector)
            y_data.append(row['label'])
        except Exception as e:
            print(f"Ошибка загрузки {file_path}: {e}")
            
    return np.array(X_data), np.array(y_data)

print("Загрузка спектрограмм в память (это может занять время)...")
X_raw, y = load_spectrograms(df_sample, BASE_PATH)
print(f"Данные загружены. Форма матрицы X: {X_raw.shape}")

# Освобождаем память
del df, sample_real, sample_fake
gc.collect()

Предобработка признаков

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(" Применение PCA...")
pca = PCA(n_components=0.95, random_state=42)
X_final = pca.fit_transform(X_scaled)

print(f"PCA завершено. Новая размерность: {X_final.shape[1]} признаков (было {X_raw.shape[1]})")
print(f"️ Объясненная дисперсия: {sum(pca.explained_variance_ratio_)*100:.2f}%")

# Удаляем тяжелые массивы
del X_raw, X_scaled
gc.collect()

Разведочныц анализ и разбиение

In [ ]:
print("Визуализация спектрограммы:")

sample_filename = df_sample.iloc[0]['filepath']
sample_label = df_sample.iloc[0]['label']

del df_sample
gc.collect()

sample_path = os.path.join(BASE_PATH, sample_filename)
spec_example = np.load(sample_path)

# Визуализация
plt.figure(figsize=(10, 4))
plt.imshow(spec_example, aspect='auto', cmap='magma')
plt.title(f"Пример спектрограммы (Класс: {sample_label})") # Используем сохраненную переменную
plt.xlabel("Time Frames")
plt.ylabel("Mel Bins")
plt.colorbar()
plt.tight_layout()
plt.show()

# 80% данных на обучение, 20% на тест
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Сохраняем пропорции классов
)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

Обучение моделей

In [ ]:
# Создаем словарь с моделями
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (Linear)': SVC(kernel='linear', random_state=42) # Linear быстрее для больших данных
}

results = {}

print("🚀 Начало обучения моделей...\n")

for name, model in models.items():
    print(f"Обучение: {name}")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    
    # Метрики
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'f1': f1,
        'auc': auc,
        'y_pred': y_pred
    }
    
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}\n")

Оценка и кросс-валидация

In [ ]:
results_df = pd.DataFrame({k: v for k, v in results.items() if k not in ['model', 'y_pred']}).T
results_df = results_df.sort_values('f1', ascending=False)

print("Таблица результатов (отсортирована по F1-score):")
print(results_df.round(4))

# Визуализация метрик
results_df[['accuracy', 'f1', 'auc']].plot(kind='bar', figsize=(10, 6))
plt.title("Сравнение метрик моделей")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Кросс-валидация
print("\nПроведение кросс-валидации")
cv_results = {}

for name, model_dict in results.items():
    model = model_dict['model']
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    cv_results[name] = scores.mean()
    print(f"{name}: Mean F1 (CV) = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

Отчеты по классификации

In [ ]:
best_model_name = results_df.index[0]
best_model_obj = results[best_model_name]['model']
best_y_pred = results[best_model_name]['y_pred']

print(f"Лучшая модель: {best_model_name}")

print("\nClassification Report:")
print(classification_report(y_test, best_y_pred, target_names=['Real (0)', 'Fake (1)']))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, best_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'Fake'])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()